# ERC circularité · 07 · Composition des pièces d'une nomenclature

À partir d'une liste de pièces (code article et description), ce notebook propose pour chacune sa **classe matière**, celle qui fixe le prix du recycleur et la filière. C'est une entrée du reste du dossier : une classe validée ici devient le type de déchet de la pièce dans les autres notebooks.

**Comment ça marche.** Chaque source (une règle sur la description, une pièce validée qui lui ressemble, le classement antérieur, un document du fabricant, un modèle de langage) *affirme* une classe. Aucune n'est crue sur parole : le **cortex** garde ces affirmations, les compare aux validations faites par une personne, et en déduit ce que vaut chaque type de source. La probabilité d'une classe combine les affirmations selon cette fiabilité.

**Le moins d'appels possible.** Sans classe, une pièce n'a pas de bac : elle ne peut pas être triée. L'**enjeu** d'une pièce est ce que rapporterait au plus, par an, d'établir sa classe : masse annuelle × moyenne de (prix du recycleur + traitement évité) sur les classes possibles. La chaîne commence par ce qui est gratuit. Une recherche de documents ou un appel à un modèle n'est lancé que si l'enjeu couvre ce qui a déjà été dépensé pour la pièce, plus cette action. Une recherche déjà payée n'est jamais refaite.

**La confiance se gagne.** Tant qu'aucune validation n'a montré qu'un modèle lit juste, sa lecture d'un document ne suffit pas à accepter une classe, même relue par lui. Elle y suffit une fois que les validations l'ont montré.

**Confidentialité.** Aucun code article interne ni aucune description ne sort du poste. Une recherche ne contient que le fabricant, sa référence et un type générique, et n'est émise que si le fabricant est nommé, n'est pas l'entreprise, et si la référence n'est le code article d'aucune pièce. Une pièce déclarée de conception interne n'est jamais recherchée. À vérifier avant de brancher une collecte : la colonne « référence fabricant » de la nomenclature contient bien des références de fabricant.

Vocabulaire : « collecte » désigne ici la recherche de documents sur le web ; « scrapé » reste réservé aux pièces mises au scrap.

In [1]:
import sys
from pathlib import Path

# Le dossier est celui qui contient le code (erc/) : le parent de notebooks/.
dossier = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "erc" / "__init__.py").is_file()), Path.cwd().parent)
sys.path.insert(0, str(dossier))
import erc
from erc import affichage as aff

ctx = erc.executer(jusqua="ligne")

ERC circularité — DÉMONSTRATION (données fictives)


  étape sources       3 artefacts → artefacts/00_sources/


  étape classement   13 artefacts → artefacts/01_classement/


  étape flux         16 artefacts → artefacts/02_flux/


  étape fiabilite    11 artefacts → artefacts/03_fiabilite/


  étape rex          13 artefacts → artefacts/04_rex/


  étape ligne        17 artefacts → artefacts/05_ligne/


In [2]:
from erc import composition
compo = composition.executer(ctx)      # nomenclature du dossier ; à défaut, les pièces consommées dans les extractions
aff.etat(ctx)

DÉMONSTRATION : toutes les données affichées sont fictives. Période : du 01/06/2026 au 01/11/2026, 154 jours extraits. Aucune anomalie de calcul.

## 1. Composition proposée

Une ligne par pièce : la classe la plus probable, sa probabilité, les sources qui l'affirment, et un statut.

| Statut | Sens |
|---|---|
| validée | une personne a fixé la classe : c'est la vérité du dossier |
| acceptée | la classe est probable (seuil d'acceptation) et une erreur coûterait moins qu'une revue ; contrôlée par sondage |
| à valider | des preuves existent mais ne suffisent pas, ou la classe est probable mais une erreur coûterait plus qu'une revue |
| inconnue | aucune source ne s'est prononcée |

**Lecture** : « Enjeu » borne ce qu'établir la classe peut rapporter par an. « Prochaine action » dit quoi faire, y compris « aucune » quand établir la classe coûterait plus que cela ne rapporte. Une classe affirmée hors du référentiel des grades apparaît comme « hors référentiel » : elle demande une revue.

**Limite** : en démonstration, la nomenclature, les documents et le modèle sont fictifs.

In [3]:
aff.table(ctx, "composition", "Composition proposée", colonnes=["PN", "Description", "Classe retenue", "Probabilité", "Statut", "Sources", "Enjeu (€/an, borne)", "Appels au modèle", "Prochaine action"])

PN,Description,Classe retenue,Probabilité,Statut,Sources,"Enjeu (€/an, borne)",Appels au modèle,Prochaine action
P-1001,IGBT 400A 1200V,Composant de puissance,"1,00",validée,validation humaine,0,0,aucune
P-1002,IGBT 400A 600V,Composant de puissance,"1,00",acceptée,"analogie, classement antérieur, règle",1 380,0,aucune : contrôle par sondage
P-1003,AL Electrolytic Capacitor 450V 4700uF 105degC,Condensateur,"1,00",acceptée,"classement antérieur, règle",,0,aucune : contrôle par sondage
P-1004,Gate Driver Board,Carte B,"1,00",validée,validation humaine,0,0,aucune
P-1005,Control board assembly,Carte B,"1,00",acceptée,"classement antérieur, règle",1 431,0,aucune : contrôle par sondage
P-1006,Bus resistance 33k ohm 10W,Composant standard,"1,00",acceptée,"classement antérieur, règle","45,69",0,aucune : contrôle par sondage
P-1007,Protection fuse 100A high speed,Composant standard,"1,00",validée,validation humaine,0,0,aucune
P-2001,Power supply board,Carte B,"1,00",validée,validation humaine,0,0,aucune
P-2002,Cooling fan 120mm,Composant standard,"1,00",acceptée,"classement antérieur, règle","77,23",0,aucune : contrôle par sondage
P-3001,Coque boîtier gris,Plastique,"1,00",validée,validation humaine,0,0,aucune


## 2. File de revue

Les pièces à faire valider par une personne, par enjeu décroissant.

**Le plus simple : la page de revue.** `artefacts/composition/ERC_Composition_Revue.html` s'ouvre dans un navigateur, sans connexion. Elle montre pour chaque pièce la classe proposée, qui dit quoi, et ce que rapporterait de trancher. La personne confirme ou corrige, puis télécharge un fichier de validations à déposer dans `Acquisition/` (dossier des extractions). Deux files y sont proposées : les pièces **à valider**, et un **sondage** de pièces acceptées tirées au hasard. Le sondage compte autant que la file : sans lui, la fiabilité des sources n'est mesurée que sur les cas douteux, et elle est sous-estimée.

**À la main.** Remplir « Classe validée » et « Validateur » dans la table ci-dessous et la déposer sous `Acquisition/validations_composition.csv` (colonnes `PN;Classe validée;Validateur;Commentaire;Motif`, le motif valant `file` pour ces pièces). Tous les fichiers `validations_composition*.csv` du dossier sont lus : une personne ou une séance par fichier.

À l'exécution suivante, ces validations entrent dans le cortex : elles fixent la classe des pièces validées, servent d'exemples aux pièces de description voisine et mesurent ce que vaut chaque type de source.

In [4]:
aff.table(ctx, "composition", "File de revue", colonnes=["PN", "Description", "Classe retenue", "Probabilité", "Enjeu (€/an, borne)", "Vaut une revue", "Prochaine action"])

PN,Description,Classe retenue,Probabilité,"Enjeu (€/an, borne)",Vaut une revue,Prochaine action
P-5001,Coil preamp board,Carte B,"0,577","78,41",oui,revue humaine


## 3. Ce que valent les sources

Pour chaque type de source : l'a priori de départ, qui est une hypothèse de travail, puis ce qui est observé sur les pièces validées. Plus il y a de validations, plus la fiabilité repose sur les faits et moins sur l'hypothèse (« Poids des observations »).

**Lecture** : une source souvent contredite par les validations voit sa fiabilité baisser, et pèse moins dans toutes les probabilités. Un modèle de langage est une source comme une autre. Les colonnes « dont par sondage » et « dont dans la file » disent d'où viennent les validations : celles de la file ne renseignent que les pièces non tranchées, celles du sondage que les pièces acceptées.

**Limite** : le calcul suppose que les types de source se trompent indépendamment. Si les règles saisies ici reprennent celles qui ont produit le classement antérieur, les deux comptent en double : ne garder que l'une des deux.

In [5]:
aff.table(ctx, "composition", "Fiabilité des sources")

Type de source,A priori (hypothèse),Affirmations vérifiées,Justes,Fausses,dont par sondage,dont dans la file,Fiabilité,Poids des observations
analogie,"0,650",0,0,0,0,0,"0,650","0,000"
classement antérieur,"0,700",6,6,0,0,0,"0,880","0,600"
document,"0,800",0,0,0,0,0,"0,800","0,000"
déclaration fabricant,"0,950",0,0,0,0,0,"0,950","0,000"
modèle : extraction,"0,600",0,0,0,0,0,"0,600","0,000"
modèle : extraction confirmée,"0,750",0,0,0,0,0,"0,750","0,000"
modèle : validation,"0,700",0,0,0,0,0,"0,700","0,000"
règle,"0,750",12,10,2,0,0,"0,812","0,750"


## 4. Journal des acquisitions

Chaque recherche de documents et chaque appel au modèle, avec son coût supposé. Une table courte est le signe que la chaîne s'arrête tôt. Une panne (collecte ou modèle) y figure comme « échec » : elle n'est pas comptée et l'action sera reprise au passage suivant.

In [6]:
aff.table(ctx, "composition", "Journal des acquisitions")

cree_le,domaine,sujet,action,cout,resultat
2026-11-26T12:06:50,composition,P-5001,collecte,"0,020",1 document(s)
2026-11-26T12:06:50,composition,P-5001,lecture par le modèle,"0,050",Carte C
2026-11-26T12:06:50,composition,P-6001,collecte,"0,020",2 document(s)
2026-11-26T12:06:50,composition,P-6001,lecture par le modèle,"0,050",Carte B
2026-11-26T12:06:50,composition,P-6001,lecture par le modèle,"0,050",Composant de puissance


## 5. Valeur d'établir une classe, dans le modèle de ligne

Pour les pièces à valider qui figurent dans le modèle de ligne : ce que rapporterait d'établir leur classe, calculé en résolvant le modèle sous chaque classe possible, comparé au modèle où la pièce ne peut pas être triée. « La décision change » dit si la réponse modifierait le plan. Table vide = aucune pièce du modèle de ligne n'attend de validation.

La justification est dans `docs/Formal_framework.md`, section 8.3 (proposition 7).

In [7]:
aff.table(ctx, "composition", "Valeur de l'information")

Pièce,Types de FRU,Classes possibles,Borne (€/an),Valeur d'établir la classe (€/an),La décision change,Calcul exact,À acquérir


**Illustration.** Le même calcul sur une pièce du modèle de ligne, en supposant que sa classe hésite entre les deux grades les mieux payés du référentiel. C'est une supposition faite ici pour montrer le calcul, pas un résultat du dossier.

In [8]:
from erc import information
import pandas as pd
m = ctx.res.get("ligne", {}).get("Modèle")
prix = {k: float(v) for k, v in ctx.src.prix_recycleur.items()}
if m and len(prix) >= 2:
    piece = next((pc for d in m["frus"].values() for pc, e in d["pieces"].items() if pc != "CARCASSE" and e.get("poids_kg") is not None), None)
    deux = sorted(prix, key=prix.get, reverse=True)[:2]
    exemple = information.priorites_classes(m["ligne"], m["frus"], {piece: {deux[0]: 0.6, deux[1]: 0.4}}, prix, cout_information=composition.Reglages().cout_revue) if piece else pd.DataFrame()
    display(exemple)
else:
    print("Modèle de ligne ou référentiel des grades absent : pas d'illustration.")

,Pièce,Types de FRU,Classes possibles,Borne (€/an),Valeur d'établir la classe (€/an),La décision change,Calcul exact,À acquérir
0,P-1004,FRU-A100-R,"Carte B 60%, Carte C 40%",780.0,525.933778,True,oui,True


## 6. Brancher une collecte de documents et un modèle local

Sans collecteur ni modèle, la chaîne s'arrête aux sources gratuites et dit ce qui manque. Les deux se branchent sans modifier le dossier :

```python
from erc import composition

def ma_collecte(requete):
    # requete = {"fabricant": ..., "reference_fabricant": ..., "type": ...} : rien d'interne
    # rendre une liste de documents : {"origine": "site ou URL", "texte": "...", "nature": "document" ou "déclaration fabricant"}
    # ajouter "classe": "..." quand le document donne lui-même la classe (déclaration matière structurée) : aucun modèle n'est alors appelé
    return []

qwen = composition.ModeleLocal("http://localhost:11434/v1", "nom-du-modele")   # tout serveur compatible « chat completions », sur ce poste
reglages = composition.Reglages(fabricants_internes=("Nom de l'entreprise",))  # jamais recherché à l'extérieur
compo = composition.executer(ctx, collecteur=ma_collecte, modele=qwen, reglages=reglages)
# un second modèle pour contrôler les lectures du premier : composition.executer(..., modele_controle=autre_modele)
```

Le modèle répond sous contrainte : la classe doit être dans la liste proposée, et la citation qui la justifie doit se trouver mot pour mot dans le document, sinon la réponse est rejetée. Il doit tourner sur ce poste : pour un serveur distant, il faut l'accepter explicitement (`distant=True`), car le texte des documents lui est envoyé.

Votre méthode de collecte existante se branche telle quelle derrière `ma_collecte` : il suffit qu'elle rende des documents sous cette forme.

In [9]:
aff.notes(ctx, "composition")

Élément,Valeur
pieces,21
statuts › acceptée,14
statuts › validée,6
statuts › à valider,1
appels au modele,0
cout des acquisitions eur,0
pieces sans appel payant,21
pieces hors referentiel,0
codes article repetes ignores,0
classes,"Carte B, Carte C, Composant de puissance, Condensateur, Câbles, Composant standard, Plastique, Métal"


---\nCe notebook est une entrée du dossier : relancer **00_Demarrage** après avoir validé des pièces pour que leurs classes soient reprises partout.